# Phase 3 — Text Classification: Fiction vs Non-Fiction

The raw `categories` column has hundreds of inconsistent tags. Instead of trying to clean them manually, we use **zero-shot classification** with `facebook/bart-large-mnli` (a free Hugging Face model) to assign each book a clean binary label:
- `fiction`
- `non-fiction`

This creates a usable filter for the Gradio dashboard.

**Prerequisites:** `1-data-exploration.ipynb` → `data/books_cleaned.csv`

**Output:** `data/books_with_categories.csv`

In [ ]:
import pandas as pd
from transformers import pipeline

## 1. Load cleaned data

In [ ]:
df = pd.read_csv('../data/books_cleaned.csv')
print(f'Loaded {len(df)} books')

## 2. Load the zero-shot classifier

`facebook/bart-large-mnli` is trained on natural language inference (NLI). Zero-shot classification works by framing each label as a hypothesis: _"This book is about fiction"_ and scoring which hypothesis the description entails more strongly.

In [ ]:
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)

candidate_labels = ['fiction', 'non-fiction']

## 3. Try it on a single example

In [ ]:
sample = df['description'].iloc[0]
print('Description:', sample[:200], '...')
result = classifier(sample, candidate_labels)
print('\nClassification result:')
for label, score in zip(result['labels'], result['scores']):
    print(f'  {label}: {score:.3f}')

## 4. Classify all books

> ⏱ **Expected time:** ~20–40 minutes on CPU for ~6,000 books. Run once and save the output.

In [ ]:
def classify_book(description: str) -> str:
    """Return 'fiction' or 'non-fiction' for a book description."""
    result = classifier(description, candidate_labels)
    return result['labels'][0]  # highest-scoring label

print('Classifying books... (this takes a while on CPU)')
df['simple_category'] = df['description'].apply(classify_book)

print('\nCategory distribution:')
print(df['simple_category'].value_counts())

## 5. Spot-check the results

In [ ]:
# A few known fiction books
df[df['simple_category'] == 'fiction'][['title', 'authors', 'categories', 'simple_category']].sample(5)

In [ ]:
# A few known non-fiction books
df[df['simple_category'] == 'non-fiction'][['title', 'authors', 'categories', 'simple_category']].sample(5)

## 6. Save the enriched dataset

In [ ]:
df.to_csv('../data/books_with_categories.csv', index=False)
print(f'Saved to data/books_with_categories.csv')